In [ ]:
# 安装 d2l（Dive into Deep Learning）工具包。
# --no-deps: 不额外安装依赖（依赖通常在环境中已具备，避免重复安装）
# --quiet: 减少安装输出，让笔记本界面更干净
!pip install d2l --no-deps --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.8 MB/s eta 0:00:00


In [ ]:
# ===== 1. 导入本节需要的库 =====
import os  # 处理文件路径，例如拼接 vec.txt 的完整路径
import torch  # PyTorch 张量计算库，用于向量运算
from torch import nn  # 神经网络模块（本节中不直接使用，但保留常见导入习惯）
from d2l import torch as d2l  # d2l 封装的数据下载与工具函数

In [ ]:
# ===== 2. 在 d2l 的数据仓库中注册要用到的数据集 =====
# 说明：d2l.DATA_HUB 是一个“名称 -> (下载地址, sha1校验值)”的字典。
# 后续调用 d2l.download_extract(名称) 时，会自动下载并解压。

#@save
# glove.6b.50d: GloVe 6B 语料训练，50维词向量（体积较小，适合教学演示）
d2l.DATA_HUB['glove.6b.50d'] = (d2l.DATA_URL + 'glove.6B.50d.zip',
                                '0b8703943ccdb6eb788e6f091b8946e82231bc4d')

#@save
# glove.6b.100d: 同样是 6B 语料，但向量维度为 100，表达能力更强、体积更大
d2l.DATA_HUB['glove.6b.100d'] = (d2l.DATA_URL + 'glove.6B.100d.zip',
                                 'cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a')

#@save
# glove.42b.300d: 42B 语料 + 300维向量，质量通常更高，但资源开销也更大
d2l.DATA_HUB['glove.42b.300d'] = (d2l.DATA_URL + 'glove.42B.300d.zip',
                                  'b5116e234e9eb9076672cfeabf5469f3eec904fa')

#@save
# wiki.en: 英文维基百科语料压缩包（用于某些 NLP 任务的数据准备）
d2l.DATA_HUB['wiki.en'] = (d2l.DATA_URL + 'wiki.en.zip',
                           'c1816da3821ae9f43899be655002f6c723e91b88')

In [ ]:
#@save
class TokenEmbedding:
    """
    使用预训练词向量文件（如 GloVe）构建“词 -> 向量”的查询工具。

    参数
    ----
    embedding_name : str
        在 d2l.DATA_HUB 中注册的名称，例如 'glove.6b.50d'。

    属性
    ----
    idx_to_token : List[str]
        下标到词的映射表。第 0 项固定为 '<unk>'（未知词）。
    idx_to_vec : torch.Tensor
        每个词对应的向量矩阵，形状约为 [词表大小, 向量维度]。
    token_to_idx : Dict[str, int]
        词到下标的映射，便于快速查找。
    unknown_idx : int
        未知词对应下标，固定为 0。
    """

    def __init__(self, embedding_name):
        # 读取词表和向量矩阵
        self.idx_to_token, self.idx_to_vec = self._load_embedding(embedding_name)

        # 未知词索引，遇到不在词表中的词时使用
        self.unknown_idx = 0

        # 构建反向字典：token -> index
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

    def _load_embedding(self, embedding_name):
        """
        读取向量文件 vec.txt，并构建词表与向量矩阵。

        返回
        ----
        idx_to_token : List[str]
            词表，首个元素为 '<unk>'。
        idx_to_vec : torch.Tensor
            与词表对齐的向量矩阵，第 0 行为全 0 向量（对应 '<unk>'）。
        """
        # 先手动放入未知词占位符
        idx_to_token, idx_to_vec = ['<unk>'], []

        # 下载并解压数据，得到目录路径
        data_dir = d2l.download_extract(embedding_name)

        # 词向量来源：
        # GloVe: https://nlp.stanford.edu/projects/glove/
        # fastText: https://fasttext.cc/
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            for line in f:
                # 每行格式：token v1 v2 ... vd
                elems = line.rstrip().split(' ')
                token, elems = elems[0], [float(elem) for elem in elems[1:]]

                # 过滤掉首行标题（某些 fastText 文件第一行不是词向量）
                if len(elems) > 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)

        # 在最前面补一行全 0 向量，和 '<unk>' 对齐
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec

        # 转成 torch.Tensor，便于后续向量计算
        return idx_to_token, torch.tensor(idx_to_vec)

    def __getitem__(self, tokens):
        """
        按“词列表”查询对应词向量。

        参数
        ----
        tokens : List[str]
            输入词列表，例如 ['man', 'woman']。

        返回
        ----
        vecs : torch.Tensor
            形状为 [len(tokens), 向量维度] 的张量。
        """
        # 对每个词查索引；找不到则回退到 unknown_idx
        indices = [self.token_to_idx.get(token, self.unknown_idx) for token in tokens]

        # 按索引取向量
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs

    def __len__(self):
        """返回词表大小（含 '<unk>'）。"""
        return len(self.idx_to_token)

In [ ]:
# 创建 50 维 GloVe 词向量对象
glove_6b50d = TokenEmbedding('glove.6b.50d')

# 查看词表大小（通常是 40 万词 + 1 个 <unk>）
print(len(glove_6b50d))

400001


In [ ]:
# 验证词和索引是否能对应上：
# 1) token_to_idx['beautiful'] 给出 beautiful 的下标；
# 2) idx_to_token[3367] 取出下标 3367 对应的词。
# 若一切正常，这两者应能相互对应。
glove_6b50d.token_to_idx['beautiful'], glove_6b50d.idx_to_token[3367]

(3367, 'beautiful')

In [ ]:
def knn(W, x, k):
    """
    在词向量矩阵 W 中，找到与向量 x 最相似的 k 个向量下标。

    参数
    ----
    W : torch.Tensor
        候选向量矩阵，形状 [n, d]。
        n 是候选数量（词表大小），d 是向量维度。
    x : torch.Tensor
        查询向量，形状可为 [1, d] 或 [d]。
    k : int
        需要返回的最相似向量个数。

    返回
    ----
    topk : torch.Tensor
        最相似的 k 个向量在 W 中的下标。
    scores : List[torch.Tensor]
        对应的余弦相似度分数。
    """
    # 1) 将 x 拉平成一维向量，便于与 W 做矩阵-向量乘法
    x = x.reshape(-1,)

    # 2) 分子：W 每行与 x 的点积
    numerator = torch.mv(W, x)

    # 3) 分母：||W_i|| * ||x||（余弦相似度公式）
    # 加 1e-9 是为了避免除以 0，增强数值稳定性
    denominator = (
        torch.sqrt(torch.sum(W * W, axis=1) + 1e-9) *
        torch.sqrt((x * x).sum())
    )

    # 4) 逐行计算余弦相似度
    cos = numerator / denominator

    # 5) 取相似度最大的前 k 个下标
    _, topk = torch.topk(cos, k=k)

    # 6) 返回下标和对应分数
    return topk, [cos[int(i)] for i in topk]

In [ ]:
def get_similar_tokens(query_token, k, embed):
    """
    打印与 query_token 最相似的 k 个词。

    参数
    ----
    query_token : str
        要查询的中心词，例如 'baby'。
    k : int
        需要输出的相似词个数。
    embed : TokenEmbedding
        词向量对象，内部包含词表和向量矩阵。

    说明
    ----
    - knn 会返回 k+1 个，因为最相似的通常是词本身；
    - 因此输出时要跳过第 1 个结果（输入词自己）。
    """
    # 取前 k+1 个：多取一个是为了排除 query_token 自身
    topk, cos = knn(embed.idx_to_vec, embed[[query_token]], k + 1)

    # topk[0] 基本就是 query_token 本身，因此从 topk[1:] 开始打印
    for i, c in zip(topk[1:], cos[1:]):
        print(f'{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}')

In [ ]:
# 示例1：查看与 baby 最相近的 3 个词
get_similar_tokens('baby', 3, glove_6b50d)

babies：cosine相似度=0.839
boy：cosine相似度=0.800
girl：cosine相似度=0.792


In [ ]:
# 示例2：查看与 beautiful 最相近的 3 个词
get_similar_tokens('beautiful', 3, glove_6b50d)

lovely：cosine相似度=0.921
gorgeous：cosine相似度=0.893
wonderful：cosine相似度=0.830


## 词语类比任务（A:B = C:?）

核心思想：若词向量学到了语义关系，那么向量差也会编码这种关系。

常见公式：
- 已知 `token_a : token_b = token_c : ?`
- 计算目标向量：`x = vec(token_b) - vec(token_a) + vec(token_c)`
- 在词表中找与 `x` 最相近的词，作为答案。

In [ ]:
def get_analogy(token_a, token_b, token_c, embed):
    """
    计算词语类比：token_a : token_b = token_c : ?

    参数
    ----
    token_a, token_b, token_c : str
        三个已知词。
        例如 token_a='man', token_b='woman', token_c='son'。
    embed : TokenEmbedding
        词向量查询对象。

    返回
    ----
    result_token : str
        预测得到的第 4 个词。

    计算步骤
    --------
    1) 取出三个词的向量：a, b, c
    2) 按类比公式构造目标向量：x = b - a + c
    3) 在整个词表中找与 x 最相近的词
    """
    # 一次取出三个词向量，vecs 形状约为 [3, d]
    vecs = embed[[token_a, token_b, token_c]]

    # 类比公式：b - a + c
    x = vecs[1] - vecs[0] + vecs[2]

    # 在词表中检索与 x 最相近的 1 个词
    topk, cos = knn(embed.idx_to_vec, x, 1)

    # 返回预测词
    return embed.idx_to_token[int(topk[0])]

In [ ]:
# 示例1：性别关系迁移
# man : woman = son : ?  期望接近 daughter
get_analogy('man', 'woman', 'son', glove_6b50d)

'daughter'

In [ ]:
# 示例2：国家-首都关系迁移
# beijing : china = tokyo : ?  期望接近 japan
get_analogy('beijing', 'china', 'tokyo', glove_6b50d)

'japan'

In [ ]:
# 示例3：形容词比较级/最高级关系迁移
# bad : worst = big : ?  期望接近 biggest
get_analogy('bad', 'worst', 'big', glove_6b50d)

'biggest'

In [ ]:
# 示例4：动词现在式-过去式关系迁移
# do : did = go : ?  期望接近 went
get_analogy('do', 'did', 'go', glove_6b50d)

'went'